In [2]:
import pandas as pd
import csv

df = pd.read_csv('data_sources.csv')
# df.sort_values(by=['Data'])
df

,Source,Data
0,inm,precip(h/j)
1,inm,temp min max(h)
2,inm,humid rel(h)
3,inm,windspeed(h)
4,inm,press atm(h)
5,openmeteo,precip(h/j)
6,openmeteo,prev meteo 7j(j)
7,noaa,precip(h/j)
8,noaa,windspeed(h)
9,openweather,temp min max(h)


In [ ]:
df.to_csv('output.csv', index=False)

# nasa

In [ ]:
import requests
import pandas as pd

url = "https://power.larc.nasa.gov/api/temporal/daily/point"

params = {
    "parameters": "T2M_MIN,T2M_MAX,PRECTOTCORR",
    "community": "AG",
    "longitude": 10.18,
    "latitude": 36.80,
    "start": "20240101",
    "end": "20241231",
    "format": "JSON"
}

r = requests.get(url, params=params)
data = r.json()["properties"]["parameter"]

df = pd.DataFrame(data)
df.index = pd.to_datetime(df.index)
df = df.sort_index()
df
# print(df.head())


,T2M_MIN,T2M_MAX,PRECTOTCORR
2024-01-01,8.68,19.04,1.03
2024-01-02,9.52,16.84,1.02
2024-01-03,7.08,19.62,0.00
2024-01-04,9.06,19.23,0.00
2024-01-05,8.18,18.90,0.00
...,...,...,...
2024-12-27,9.63,14.88,9.11
2024-12-28,9.98,15.71,1.57
2024-12-29,9.41,16.47,0.39
2024-12-30,7.82,15.94,0.04


In [ ]:
print(1)

1


# foa

In [ ]:
df = pd.read_csv("/content/Prices_E_All_Data.csv", low_memory=False)

year_cols = [c for c in df.columns if c.startswith("Y") and not c.endswith("F")]
flag_cols = {c[:-1]: c for c in df.columns if c.endswith("F")}
# year_cols

In [ ]:
df_long = df.melt(
    id_vars=[
        "Area Code", "Area",
        "Item Code", "Item",
        "Element Code", "Element",
        "Months Code", "Months",
        "Unit"
    ],
    value_vars=year_cols,
    var_name="Year",
    value_name="Value"
)

df_long["Year"] = df_long["Year"].str[1:].astype(int)
df_long

,Area Code,Area,Item Code,Item,Element Code,Element,Months Code,Months,Unit,Year,Value
0,2,Afghanistan,221,"Almonds, in shell",5530,Producer Price (LCU/tonne),7021,Annual value,LCU,1991,NaN
1,2,Afghanistan,221,"Almonds, in shell",5531,Producer Price (SLC/tonne),7021,Annual value,SLC,1991,NaN
2,2,Afghanistan,221,"Almonds, in shell",5539,Producer Price Index (2014-2016 = 100),7021,Annual value,NaN,1991,21.57
3,2,Afghanistan,711,"Anise, badian, coriander, cumin, caraway, fenn...",5530,Producer Price (LCU/tonne),7021,Annual value,LCU,1991,NaN
4,2,Afghanistan,711,"Anise, badian, coriander, cumin, caraway, fenn...",5531,Producer Price (SLC/tonne),7021,Annual value,SLC,1991,NaN
...,...,...,...,...,...,...,...,...,...,...,...
3540177,181,Zimbabwe,1732,"Oilcrops, Oil Equivalent",5539,Producer Price Index (2014-2016 = 100),7021,Annual value,NaN,2024,NaN
3540178,181,Zimbabwe,1726,"Pulses, Total",5539,Producer Price Index (2014-2016 = 100),7021,Annual value,NaN,2024,NaN
3540179,181,Zimbabwe,1720,"Roots and Tubers, Total",5539,Producer Price Index (2014-2016 = 100),7021,Annual value,NaN,2024,NaN
3540180,181,Zimbabwe,1735,Vegetables Primary,5539,Producer Price Index (2014-2016 = 100),7021,Annual value,NaN,2024,NaN


In [ ]:
flags = df.melt(
    id_vars=[
        "Area Code", "Area",
        "Item Code", "Item",
        "Element Code", "Element",
        "Months Code", "Months"
    ],
    value_vars=list(flag_cols.values()),
    var_name="YearF",
    value_name="Flag"
)

flags["Year"] = flags["YearF"].str[1:-1].astype(int)

df_long = df_long.merge(
    flags.drop(columns="YearF"),
    on=[
        "Area Code", "Area",
        "Item Code", "Item",
        "Element Code", "Element",
        "Months Code", "Months",
        "Year"
    ],
    how="left"
)
df_long

,Area Code,Area,Item Code,Item,Element Code,Element,Months Code,Months,Unit,Year,Value,Flag
0,2,Afghanistan,221,"Almonds, in shell",5530,Producer Price (LCU/tonne),7021,Annual value,LCU,1991,NaN,NaN
1,2,Afghanistan,221,"Almonds, in shell",5531,Producer Price (SLC/tonne),7021,Annual value,SLC,1991,NaN,NaN
2,2,Afghanistan,221,"Almonds, in shell",5539,Producer Price Index (2014-2016 = 100),7021,Annual value,NaN,1991,21.57,I
3,2,Afghanistan,711,"Anise, badian, coriander, cumin, caraway, fenn...",5530,Producer Price (LCU/tonne),7021,Annual value,LCU,1991,NaN,NaN
4,2,Afghanistan,711,"Anise, badian, coriander, cumin, caraway, fenn...",5531,Producer Price (SLC/tonne),7021,Annual value,SLC,1991,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
3540177,181,Zimbabwe,1732,"Oilcrops, Oil Equivalent",5539,Producer Price Index (2014-2016 = 100),7021,Annual value,NaN,2024,NaN,NaN
3540178,181,Zimbabwe,1726,"Pulses, Total",5539,Producer Price Index (2014-2016 = 100),7021,Annual value,NaN,2024,NaN,NaN
3540179,181,Zimbabwe,1720,"Roots and Tubers, Total",5539,Producer Price Index (2014-2016 = 100),7021,Annual value,NaN,2024,NaN,NaN
3540180,181,Zimbabwe,1735,Vegetables Primary,5539,Producer Price Index (2014-2016 = 100),7021,Annual value,NaN,2024,NaN,NaN


In [ ]:
df_long_t = df_long[df_long["Area"] == "Tunisia"]
# df_long_t.reset_index(drop=True, inplace=True)
df_long_t

,Area Code,Area,Item Code,Item,Element Code,Element,Months Code,Months,Unit,Year,Value,Flag
94495,222,Tunisia,221,"Almonds, in shell",5530,Producer Price (LCU/tonne),7021,Annual value,LCU,1991,1360.00,A
94496,222,Tunisia,221,"Almonds, in shell",5531,Producer Price (SLC/tonne),7021,Annual value,SLC,1991,1360.00,A
94497,222,Tunisia,221,"Almonds, in shell",5532,Producer Price (USD/tonne),7021,Annual value,USD,1991,1470.90,A
94498,222,Tunisia,221,"Almonds, in shell",5539,Producer Price Index (2014-2016 = 100),7021,Annual value,NaN,1991,32.46,I
94499,222,Tunisia,711,"Anise, badian, coriander, cumin, caraway, fenn...",5530,Producer Price (LCU/tonne),7021,Annual value,LCU,1991,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
3530974,222,Tunisia,1726,"Pulses, Total",5539,Producer Price Index (2014-2016 = 100),7021,Annual value,NaN,2024,141.94,I
3530975,222,Tunisia,1720,"Roots and Tubers, Total",5539,Producer Price Index (2014-2016 = 100),7021,Annual value,NaN,2024,151.81,I
3530976,222,Tunisia,1729,"Treenuts, Total",5539,Producer Price Index (2014-2016 = 100),7021,Annual value,NaN,2024,134.20,I
3530977,222,Tunisia,1735,Vegetables Primary,5539,Producer Price Index (2014-2016 = 100),7021,Annual value,NaN,2024,164.04,I


In [ ]:
df_long_t['Item Code'].nunique()

119

In [ ]:
unique_items = df_long_t[['Item Code', 'Item']].drop_duplicates()
unique_items_sorted = unique_items.sort_values(by='Item Code')
unique_items_sorted.to_csv('unique_item_codes_and_items.csv', index=False)
unique_items_sorted

,Item Code,Item
94899,15,Wheat
94523,44,Barley
94719,75,Oats
94855,83,Sorghum
94883,97,Triticale
...,...,...
94909,1801,"Fruit excl Melons, Total"
94905,1804,"Citrus Fruit, Total"
94906,1814,"Coarse Grain, Total"
94911,2044,Livestock


# nasapower

In [1]:
# Test: NASA POWER range extraction validates the backfill batching solution.
# The fix should call the API once for a date range, then keep writing one partition per day.
import requests
import pandas as pd
from datetime import datetime

NASA_POWER_URL = "https://power.larc.nasa.gov/api/temporal/daily/point"
TEST_LAT = 36.8
TEST_LON = 10.2
TEST_START = "20250101"
TEST_END = "20250107"
TEST_PARAMETERS = [
    "T2M",
    "PRECTOTCORR",
    "RH2M",
    "WS2M",
    "ALLSKY_SFC_SW_DWN",
    "CLRSKY_SFC_SW_DWN",
]

range_params = {
    "start": TEST_START,
    "end": TEST_END,
    "latitude": TEST_LAT,
    "longitude": TEST_LON,
    "community": "RE",
    "parameters": ",".join(TEST_PARAMETERS),
    "format": "JSON",
    "header": "true",
}

response = requests.get(NASA_POWER_URL, params=range_params, timeout=60)
response.raise_for_status()
payload = response.json()
params_data = payload["properties"]["parameter"]

expected_dates = pd.date_range(
    start=datetime.strptime(TEST_START, "%Y%m%d"),
    end=datetime.strptime(TEST_END, "%Y%m%d"),
    freq="D",
).strftime("%Y%m%d").tolist()

missing_parameters = set(TEST_PARAMETERS) - set(params_data)
assert not missing_parameters, f"NASA response is missing parameters: {missing_parameters}"

for parameter in TEST_PARAMETERS:
    returned_dates = set(params_data[parameter])
    missing_dates = set(expected_dates) - returned_dates
    assert not missing_dates, f"{parameter} is missing dates: {sorted(missing_dates)}"

records = []
for parameter, date_values in params_data.items():
    if parameter not in TEST_PARAMETERS:
        continue
    for date_str, value in date_values.items():
        if date_str in expected_dates:
            records.append({
                "date": datetime.strptime(date_str, "%Y%m%d"),
                "parameter": parameter,
                "value": value,
            })

records_df = pd.DataFrame(records).drop_duplicates(subset=["date", "parameter"])
unique_dates = records_df["date"].dt.strftime("%Y%m%d").sort_values().unique().tolist()
assert unique_dates == expected_dates, f"Unexpected parsed dates: {unique_dates}"

expected_record_count = len(expected_dates) * len(TEST_PARAMETERS)
assert len(records_df) == expected_record_count, (
    f"Expected {expected_record_count} records, got {len(records_df)}"
)

partition_paths = records_df["date"].dt.strftime(
    "raw/daily/nasapower/year=%Y/month=%m/day=%d/data.parquet"
)
assert partition_paths.nunique() == len(expected_dates), "One output partition per day was not preserved"

old_daily_call_count = len(expected_dates)
new_range_call_count = 1
assert new_range_call_count < old_daily_call_count

print(
    f"OK: one NASA POWER call returned {len(expected_dates)} days x "
    f"{len(TEST_PARAMETERS)} parameters = {len(records_df)} records."
)
print(
    f"API calls for this range drop from {old_daily_call_count} daily calls "
    f"to {new_range_call_count} batched call."
)
records_df.head()


OK: one NASA POWER call returned 7 days x 6 parameters = 42 records.
API calls for this range drop from 7 daily calls to 1 batched call.


,date,parameter,value
0,2025-01-01,T2M,12.17
1,2025-01-02,T2M,12.68
2,2025-01-03,T2M,12.92
3,2025-01-04,T2M,13.69
4,2025-01-05,T2M,13.01


In [ ]:
LAT = 36.8      # Tunisia example
LON = 10.2
start_str = 20250101
end_str = 20251231

url = "https://power.larc.nasa.gov/api/temporal/daily/point"
params = {
"start": start_str,
"end": end_str,
"latitude": LAT,
"longitude": LON,
"community": "RE",
"parameters": "T2M,PRECTOTCORR,RH2M,WS2M,ALLSKY_SFC_SW_DWN,CLRSKY_SFC_SW_DWN",
"format": "JSON",
"header": "true"
}

        
# Appel API avec gestion des Timeouts pour éviter de bloquer indéfiniment
response = requests.get(url, params=params, timeout=30)
response.raise_for_status() # Lève une exception si le code HTTP est >= 400
data = response.json()





In [4]:
data["properties"]["parameter"]["ALLSKY_SFC_SW_DWN"]

{'20250101': 2.1862,
 '20250102': 1.7357,
 '20250103': 2.9054,
 '20250104': 2.0026,
 '20250105': 3.0041,
 '20250106': 2.5205,
 '20250107': 2.9033,
 '20250108': 3.1152,
 '20250109': 2.7866,
 '20250110': 3.0029,
 '20250111': 3.0593,
 '20250112': 1.6178,
 '20250113': 1.8329,
 '20250114': 1.4599,
 '20250115': 2.4182,
 '20250116': 2.7799,
 '20250117': 1.1746,
 '20250118': 1.5958,
 '20250119': 2.8301,
 '20250120': 3.175,
 '20250121': 3.0038,
 '20250122': 2.9784,
 '20250123': 3.0994,
 '20250124': 3.2981,
 '20250125': 2.821,
 '20250126': 3.1428,
 '20250127': 2.7818,
 '20250128': 3.2818,
 '20250129': 3.2246,
 '20250130': 3.4865,
 '20250131': 3.6607,
 '20250201': 1.5355,
 '20250202': 2.8728,
 '20250203': 2.0808,
 '20250204': 3.3866,
 '20250205': 3.1361,
 '20250206': 2.8728,
 '20250207': 1.3426,
 '20250208': 2.3506,
 '20250209': 2.9263,
 '20250210': 2.479,
 '20250211': 2.6294,
 '20250212': 4.1093,
 '20250213': 3.4771,
 '20250214': 3.283,
 '20250215': 3.179,
 '20250216': 4.3766,
 '20250217': 3.768

In [8]:
from datetime import datetime, timedelta
try:
    params_data = data["properties"]["parameter"]
except KeyError as e:
        logger.error(f"Format de réponse API inattendu : clé manquante {e}")
        raise ValueError("Structure JSON invalide de l'API NasaPower")

records = []
        
        # Invert long format (parameter -> date -> value)
for param_name, date_values in params_data.items():
            for date_str, value in date_values.items():
                records.append({
                    "date": datetime.strptime(date_str, "%Y%m%d"),
                    "parameter": param_name,
                    "value": value
                })
                

In [ ]:
records

[{'date': datetime.datetime(2025, 1, 1, 0, 0),
  'parameter': 'T2M',
  'value': 12.17},
 {'date': datetime.datetime(2025, 1, 2, 0, 0),
  'parameter': 'T2M',
  'value': 12.68},
 {'date': datetime.datetime(2025, 1, 3, 0, 0),
  'parameter': 'T2M',
  'value': 12.92},
 {'date': datetime.datetime(2025, 1, 4, 0, 0),
  'parameter': 'T2M',
  'value': 13.69},
 {'date': datetime.datetime(2025, 1, 5, 0, 0),
  'parameter': 'T2M',
  'value': 13.01},
 {'date': datetime.datetime(2025, 1, 6, 0, 0),
  'parameter': 'T2M',
  'value': 12.6},
 {'date': datetime.datetime(2025, 1, 7, 0, 0),
  'parameter': 'T2M',
  'value': 12.32},
 {'date': datetime.datetime(2025, 1, 8, 0, 0),
  'parameter': 'T2M',
  'value': 12.58},
 {'date': datetime.datetime(2025, 1, 9, 0, 0),
  'parameter': 'T2M',
  'value': 12.5},
 {'date': datetime.datetime(2025, 1, 10, 0, 0),
  'parameter': 'T2M',
  'value': 14.74},
 {'date': datetime.datetime(2025, 1, 11, 0, 0),
  'parameter': 'T2M',
  'value': 14.38},
 {'date': datetime.datetime(2025

# ECMWF

In [6]:
from ecmwfapi import ECMWFDataServer

server = ECMWFDataServer(url= "https://api.ecmwf.int/v1",
    key   = "e1552a2c45e7daa0e786f2753b9120b2",
    email = "tayeb.elleuch@ensi-uma.tn")

server.retrieve({
    "class": "ti",
    "dataset": "tigge",
    "date": "2024-09-01/to/2024-09-30",
    "expver": "prod",
    "grid": "0.5/0.5",
    "levtype": "sfc",
    "origin": "ecmf",
    "param": "176",
    "step": "0/120/240/360",
    "time": "00:00:00/12:00:00",
    "type": "cf",
    "target": "output"
})

2025-12-21 00:19:49 ECMWF API python library 1.6.5
2025-12-21 00:19:49 ECMWF API at https://api.ecmwf.int/v1
2025-12-21 00:19:50 Welcome tayeb elleuch
2025-12-21 00:19:52 In case of problems, please check https://confluence.ecmwf.int/display/WEBAPI/Web+API+FAQ or contact servicedesk@ecmwf.int
2025-12-21 00:19:52 ------------ WARNING ------------
2025-12-21 00:19:52 Access to this dataset is transitioning to a new interface, dates to be announced soon
2025-12-21 00:19:52 For more information on how to access this data in the future, visit https://confluence.ecmwf.int/x/-wUiEw
2025-12-21 00:19:52 ---------------------------------
2025-12-21 00:19:53 Request submitted
2025-12-21 00:19:53 Request id: 69472f182c1ebd563e8afea4
2025-12-21 00:19:53 Request is submitted
2025-12-21 00:19:54 Request is queued
2025-12-21 00:23:04 Calling 'nice mars /tmp/20251220-2320/33/tmp-_mars-dhY7b7-3b82773bec416724b2b8082a1af6b2ea.req'
2025-12-21 00:23:04 Forcing MIR_CACHE_PATH=/data/ec_coeff
2025-12-21 00:23

# openweather

In [ ]:
import requests
import json

# Replace with your OpenWeatherMap API key
API_KEY =";)"

# Build the request URL
url=f"https://api.openweathermap.org/data/2.5/weather?lat=36.807&lon=10.067&appid={API_KEY}"
try:
    response = requests.get(url)
    response.raise_for_status()  # Raise error for bad status codes
    weather_data = response.json()
    
    # Pretty print all the data
    print(json.dumps(weather_data, indent=4))
    with open("openweather_data.json", "w") as f:
        json.dump(weather_data, f, indent=4)
except requests.exceptions.RequestException as e:
    print(f"Error fetching weather data: {e}")


# openmeteo

In [13]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
	"latitude": 36.8089,
	"longitude": 10.064,
	"start_date": "2025-11-01",
	"end_date": "2025-12-11",
	"daily": ["temperature_2m_mean", "temperature_2m_max", "temperature_2m_min", "precipitation_sum", "rain_sum", "snowfall_sum"],
	"hourly": ["et0_fao_evapotranspiration", "direct_radiation", "soil_moisture_0_to_7cm", "temperature_2m", "precipitation", "rain", "snowfall", "relative_humidity_2m", "dew_point_2m", "soil_temperature_0_to_7cm"],
	"timezone": "auto",
}
responses = openmeteo.weather_api(url, params=params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_et0_fao_evapotranspiration = hourly.Variables(0).ValuesAsNumpy()
hourly_direct_radiation = hourly.Variables(1).ValuesAsNumpy()
hourly_soil_moisture_0_to_7cm = hourly.Variables(2).ValuesAsNumpy()
hourly_temperature_2m = hourly.Variables(3).ValuesAsNumpy()
hourly_precipitation = hourly.Variables(4).ValuesAsNumpy()
hourly_rain = hourly.Variables(5).ValuesAsNumpy()
hourly_snowfall = hourly.Variables(6).ValuesAsNumpy()
hourly_relative_humidity_2m = hourly.Variables(7).ValuesAsNumpy()
hourly_dew_point_2m = hourly.Variables(8).ValuesAsNumpy()
hourly_soil_temperature_0_to_7cm = hourly.Variables(9).ValuesAsNumpy()

hourly_data = {"date": pd.date_range(
	start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
	end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = hourly.Interval()),
	inclusive = "left"
)}

hourly_data["et0_fao_evapotranspiration"] = hourly_et0_fao_evapotranspiration
hourly_data["direct_radiation"] = hourly_direct_radiation
hourly_data["soil_moisture_0_to_7cm"] = hourly_soil_moisture_0_to_7cm
hourly_data["temperature_2m"] = hourly_temperature_2m
hourly_data["precipitation"] = hourly_precipitation
hourly_data["rain"] = hourly_rain
hourly_data["snowfall"] = hourly_snowfall
hourly_data["relative_humidity_2m"] = hourly_relative_humidity_2m
hourly_data["dew_point_2m"] = hourly_dew_point_2m
hourly_data["soil_temperature_0_to_7cm"] = hourly_soil_temperature_0_to_7cm

hourly_dataframe = pd.DataFrame(data = hourly_data)
hourly_dataframe.to_csv('daily_openmeteo.csv', index=False)
print("\nHourly data\n", hourly_dataframe)

# Process daily data. The order of variables needs to be the same as requested.
daily = response.Daily()
daily_temperature_2m_mean = daily.Variables(0).ValuesAsNumpy()
daily_temperature_2m_max = daily.Variables(1).ValuesAsNumpy()
daily_temperature_2m_min = daily.Variables(2).ValuesAsNumpy()
daily_precipitation_sum = daily.Variables(3).ValuesAsNumpy()
daily_rain_sum = daily.Variables(4).ValuesAsNumpy()
daily_snowfall_sum = daily.Variables(5).ValuesAsNumpy()

daily_data = {"date": pd.date_range(
	start = pd.to_datetime(daily.Time(), unit = "s", utc = True),
	end =  pd.to_datetime(daily.TimeEnd(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = daily.Interval()),
	inclusive = "left"
)}

daily_data["temperature_2m_mean"] = daily_temperature_2m_mean
daily_data["temperature_2m_max"] = daily_temperature_2m_max
daily_data["temperature_2m_min"] = daily_temperature_2m_min
daily_data["precipitation_sum"] = daily_precipitation_sum
daily_data["rain_sum"] = daily_rain_sum
daily_data["snowfall_sum"] = daily_snowfall_sum

daily_dataframe = pd.DataFrame(data = daily_data)
daily_dataframe.to_csv('daily_openmeteo.csv', index=False)
print("\nDaily data\n", daily_dataframe)


Coordinates: 36.8014030456543°N 10.052562713623047°E
Elevation: 42.0 m asl
Timezone: b'Africa/Tunis'b'GMT+1'
Timezone difference to GMT+0: 3600s

Hourly data
                          date  et0_fao_evapotranspiration  direct_radiation  \
0   2025-10-31 23:00:00+00:00                    0.006603               0.0   
1   2025-11-01 00:00:00+00:00                    0.003688               0.0   
2   2025-11-01 01:00:00+00:00                    0.000000               0.0   
3   2025-11-01 02:00:00+00:00                    0.000000               0.0   
4   2025-11-01 03:00:00+00:00                    0.000000               0.0   
..                        ...                         ...               ...   
979 2025-12-11 18:00:00+00:00                    0.000000               0.0   
980 2025-12-11 19:00:00+00:00                    0.000000               0.0   
981 2025-12-11 20:00:00+00:00                    0.000000               0.0   
982 2025-12-11 21:00:00+00:00                    0.

In [ ]:
h=hourly.Variables(1).ValuesAsNumpy()
h

# MODIS

In [2]:
import earthaccess
from datetime import date

# Authenticate (uses saved Earthdata credentials)
earthaccess.login()

# Search MODIS daily LST (Terra)
results = earthaccess.search_data(
    short_name="MOD11A1",
    temporal=("2022-07-15", "2022-07-15"),
    bounding_box=(7, 30, 12, 37)  # W, S, E, N (Tunisia)
)

print(f"Found {len(results)} granules")

# Download
files = earthaccess.download(results, "./modis_lst")


Found 4 granules


QUEUEING TASKS | : 100%|██████████| 4/4 [00:00<00:00, 1093.41it/s]
PROCESSING TASKS | : 100%|██████████| 4/4 [00:22<00:00,  5.66s/it]
COLLECTING RESULTS | : 100%|██████████| 4/4 [00:00<00:00, 80273.76it/s]


In [3]:
import rasterio
import numpy as np

file = files[0]  # one tile

with rasterio.open(f"HDF4_EOS:EOS_GRID:{file}:MODIS_Grid_Daily_1km_LST:LST_Day_1km") as src:
    lst = src.read(1).astype(float)

# Apply scale factor
lst[lst == 0] = np.nan
lst_celsius = lst * 0.02 - 273.15

print(np.nanmean(lst_celsius))

RasterioIOError: HDF4_EOS:EOS_GRID:modis_lst/MOD11A1.A2022196.h18v06.061.2022197092737.hdf:MODIS_Grid_Daily_1km_LST:LST_Day_1km: No such file or directory

In [6]:
import rasterio

file = files[0]

with rasterio.open(file) as ds:
    for sub in ds.subdatasets:
        print(sub)

RasterioIOError: 'modis_lst/MOD11A1.A2022196.h18v06.061.2022197092737.hdf' not recognized as being in a supported file format.

# sentinel-2

In [ ]:
from sentinelhub import (
    SHConfig, SentinelHubRequest, DataCollection,
    BBox, CRS, bbox_to_dimensions, MimeType
)

config = SHConfig()
config.sh_client_id="d4494188-b68b-4589-9ba2-6325c02f3bae"
config.sh_client_secret= "q4QZYLMrMKa0IXL4I3aQ3jFp3HPlVLWW"

bbox = BBox([7.5, 30.0, 11.5, 37.5], crs=CRS.WGS84)
size = bbox_to_dimensions(bbox, resolution=1000)

evalscript = """
//VERSION=3
function setup() {
  return {
    input: ["B04", "B08"],
    output: { bands: 1, sampleType: "FLOAT32" }
  };
}
function evaluatePixel(s) {
  return [(s.B08 - s.B04) / (s.B08 + s.B04)];
}
"""

request = SentinelHubRequest(
    evalscript=evalscript,
    input_data=[
        SentinelHubRequest.input_data(
            data_collection=DataCollection.SENTINEL2_L2A,
            time_interval=("2023-07-01", "2023-07-10"),
        )
    ],
    responses=[
        SentinelHubRequest.output_response("default", MimeType.TIFF)
    ],
    bbox=bbox,
    data_folder="ndvi_output",
    size=size,
    config=config,
)

request.save_data()

KeyError: '.npy'

In [ ]:
with rasterio.open("ndvi_output/hh/response.tiff") as src:
    ndvi = src.read(1)
    ndvi[ndvi == src.nodata] = np.nan

In [20]:
ndvi

array([[-0.0278481 , -0.02732794, -0.02865762, ..., -0.02347418,
        -0.02278083, -0.02021773],
       [-0.02851324, -0.02755102, -0.02822581, ..., -0.02303415,
        -0.02134387, -0.02107728],
       [-0.02897814, -0.02828283, -0.02702703, ..., -0.02527646,
        -0.0212766 , -0.02053712],
       ...,
       [ 0.07805765,  0.0805761 ,  0.08729192, ...,  0.09094642,
         0.08719292,  0.09678119],
       [ 0.08323281,  0.08514891,  0.08011292, ...,  0.08829234,
         0.09183674,  0.08720987],
       [ 0.08146996,  0.08141962,  0.08592322, ...,  0.09611057,
         0.08931217,  0.09168755]], shape=(834, 366), dtype=float32)

In [7]:
import pandas as pd
import os

# 1. Setup paths
home = os.path.expanduser("~")
input_path = os.path.join(home, "Downloads", "open-meteo-36.80N10.17E26m.csv")
output_path = os.path.join(home, "Downloads", "clean_daily_historical.csv")

# 2. Define your mapping (CSV_Column_Name: Schema_Column_Name)
# Update the keys below to match your ACTUAL CSV headers
column_mapping = {
    "time" : "date",
    "temperature_2m_mean (°C)" : "temperature_2m_mean",
    "temperature_2m_min (°C)" : "temperature_2m_min",
    "temperature_2m_max (°C)" : "temperature_2m_max",
    "precipitation_sum (mm)": "precipitation_sum"

}

def transform_weather_data():
    if not os.path.exists(input_path):
        print(f"Error: Could not find {input_path}")
        return

    print("Loading data...")
    df = pd.read_csv(input_path,skiprows=2)

    # 3. Rename columns using the mapping
    print("Transforming columns...")
    df = df.rename(columns=column_mapping)

    # 4. Filter to keep only the columns defined in your schema
    # This prevents 'extra' CSV columns from breaking the ClickHouse import
    schema_cols = list(column_mapping.values())
    df = df[schema_cols]

    # 5. Optional: Ensure data types are correct
    df['date'] = pd.to_datetime(df['date']).dt.date

    # 6. Save the cleaned CSV
    df.to_csv(output_path, index=False)
    print(f"Success! Cleaned file saved to: {output_path}")
    print(df)

transform_weather_data()

Loading data...
Transforming columns...
Success! Cleaned file saved to: /home/tayeb/Downloads/clean_daily_historical.csv
             date  temperature_2m_mean  temperature_2m_min  \
0      1996-02-20                 11.5                 8.4   
1      1996-02-21                  8.6                 6.0   
2      1996-02-22                  7.1                 4.7   
3      1996-02-23                  8.3                 6.2   
4      1996-02-24                 10.6                 6.6   
...           ...                  ...                 ...   
10954  2026-02-16                 14.1                11.9   
10955  2026-02-17                 14.4                12.1   
10956  2026-02-18                 13.9                 9.2   
10957  2026-02-19                 13.9                11.3   
10958  2026-02-20                 12.9                10.9   

       temperature_2m_max  precipitation_sum  
0                    17.3                2.3  
1                    12.9               

In [4]:
import pandas as pd
import os

# 1. Setup paths
home = os.path.expanduser("~")
input_path = os.path.join(home, "Downloads", "open-meteo-36.80N10.17E26m.csv")
output_path = os.path.join(home, "Downloads", "clean_daily_historical.csv")
df = pd.read_csv(input_path,skiprows=2)
df

,time,temperature_2m_mean (°C),temperature_2m_min (°C),temperature_2m_max (°C),precipitation_sum (mm)
0,1996-02-20,11.5,8.4,17.3,2.3
1,1996-02-21,8.6,6.0,12.9,0.4
2,1996-02-22,7.1,4.7,10.6,1.9
3,1996-02-23,8.3,6.2,10.5,3.3
4,1996-02-24,10.6,6.6,15.4,0.0
...,...,...,...,...,...
10954,2026-02-16,14.1,11.9,16.8,0.0
10955,2026-02-17,14.4,12.1,16.5,2.1
10956,2026-02-18,13.9,9.2,19.6,0.1
10957,2026-02-19,13.9,11.3,19.6,0.1


In [1]:
import pandas as pd
pd.read_csv("~/Documents/mock.csv")

,date,location,temp,rain
0,01-01-2026,sfax,21,3
1,02-01-2026,sfax,21,3
2,03-01-2026,tunis,22,5
3,04-01-2026,tunis,22,5
4,05-01-2026,sfax,21,3
5,06-01-2026,sfax,21,3
6,07-01-2026,tunis,22,5


In [ ]:
import google.generativeai as genai

# Configure your key
genai.configure(api_key=)

print("--- AVAILABLE MODELS ---")
for m in genai.list_models():
    # We only care about models that can generate text/chat
    if 'generateContent' in m.supported_generation_methods:
        print(m.name)

--- AVAILABLE MODELS ---
models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-